In [15]:
import requests
import json
import time
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.functions import get_json_object, col, current_timestamp, lit, sha2, concat_ws, count as spark_count, concat

# ===== SETTINGS =====
BASE_URL = "https://hapi.fhir.org/baseR4"
RECORDS_PER_PAGE = 20
RESOURCE_ORDER = ["Patient", "Encounter", "Observation", "Condition"]
TODAY_DATE = datetime.now().strftime("%Y-%m-%d")

StatementMeta(, 6450f0f5-9c23-4dcf-8a34-8452e5167583, 17, Finished, Available, Finished, False)

In [16]:
def fetch_and_load_bronze(resource_name, start_time, end_time, run_label):
    """
    Fetches data from FHIR API for a given time window (incremental load),
    saves raw JSON to Raw layer, and appends structured data to Bronze table.
    This function is reusable for ANY date range - manual or pipeline-triggered.
    """
    url = BASE_URL + "/" + resource_name
    params = {
        "_count": RECORDS_PER_PAGE,
        "_lastUpdated": ["ge" + start_time, "le" + end_time]
    }
    
    data_list = []
    page_number = 1
    current_url = url
    current_params = params
    
    while current_url is not None:
        if page_number == 1:
            response = requests.get(current_url, params=current_params)
        else:
            response = requests.get(current_url)
        
        response_data = response.json()
        extraction_time = datetime.now(timezone.utc).isoformat()
        
        data_list.append({
            "page_number": page_number,
            "extraction_timestamp": extraction_time,
            "api_url_or_params": response.url,
            "raw_data": response_data
        })
        
        next_page_url = None
        for one_link in response_data.get("link", []):
            if one_link.get("relation") == "next":
                next_page_url = one_link.get("url")
        
        current_url = next_page_url
        page_number += 1
    
    total_found = sum(len(p["raw_data"].get("entry", [])) for p in data_list)
    print(run_label, "-", resource_name, "- Records found:", total_found)
    
    if total_found == 0:
        return
    
    # Save Raw layer
    raw_folder_path = "Files/raw/" + resource_name.lower() + "/" + run_label
    for page in data_list:
        file_path = raw_folder_path + "/page_" + str(page["page_number"]) + ".json"
        mssparkutils.fs.put(file_path, json.dumps(page), overwrite=True)
    
    # Build Bronze records
    bronze_records = []
    for page in data_list:
        entries = page["raw_data"].get("entry", [])
        for entry in entries:
            resource = entry.get("resource", {})
            bronze_records.append(Row(
                resource_id=resource.get("id", ""),
                resource_json=json.dumps(resource),
                extraction_timestamp=page["extraction_timestamp"],
                api_url_or_params=page["api_url_or_params"]
            ))
    
    bronze_df = spark.createDataFrame(bronze_records)
    table_name = "bronze_" + resource_name.lower()
    bronze_df.write.format("delta").mode("append").saveAsTable(table_name)
    
    print(run_label, "-", resource_name, "- Bronze table updated:", bronze_df.count(), "records added")
    print("---------------------------------------------")

StatementMeta(, 6450f0f5-9c23-4dcf-8a34-8452e5167583, 18, Finished, Available, Finished, False)

In [17]:
FIELD_MAPPINGS = {
    "Patient": {
        "gender": "$.gender", "birth_date": "$.birthDate",
        "family_name": "$.name[0].family", "given_name": "$.name[0].given[0]",
        "active": "$.active"
    },
    "Encounter": {
        "status": "$.status", "encounter_class": "$.class.code",
        "patient_reference": "$.subject.reference"
    },
    "Observation": {
        "code_text": "$.code.text", "patient_reference": "$.subject.reference",
        "value": "$.valueQuantity.value", "unit": "$.valueQuantity.unit"
    },
    "Condition": {
        "clinical_status": "$.clinicalStatus.coding[0].code",
        "verification_status": "$.verificationStatus.coding[0].code",
        "code_text": "$.code.text", "patient_reference": "$.subject.reference"
    }
}

def build_silver_layer(resource_name):
    """
    Cleans and deduplicates Bronze data into Silver layer.
    Always rebuilds from full Bronze history, so no duplicates ever occur.
    """
    bronze_table = "bronze_" + resource_name.lower()
    silver_table = "silver_" + resource_name.lower()
    field_mappings = FIELD_MAPPINGS[resource_name]
    
    bronze_df = spark.table(bronze_table)
    silver_df = bronze_df.select("resource_id", "resource_json", "extraction_timestamp", "api_url_or_params")
    
    for column_name, json_path in field_mappings.items():
        silver_df = silver_df.withColumn(column_name, get_json_object(col("resource_json"), json_path))
    
    silver_df = silver_df.dropDuplicates(["resource_id"])
    silver_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
    
    print(resource_name, "- Silver table saved:", silver_df.count(), "clean records")
    print("---------------------------------------------")

StatementMeta(, 6450f0f5-9c23-4dcf-8a34-8452e5167583, 19, Finished, Available, Finished, False)

In [18]:
def apply_scd_type2(resource_name):
    silver_table = "silver_" + resource_name.lower()
    scd_table = "scd_" + resource_name.lower()
    
    new_data_df = spark.table(silver_table)
    hash_columns = [col(c).cast("string") for c in new_data_df.columns if c != "resource_id"]
    new_data_df = new_data_df.withColumn("record_hash", sha2(concat_ws("|", *hash_columns), 256))
    
    table_exists = spark.catalog.tableExists(scd_table)
    
    if not table_exists:
        scd_df = (new_data_df.withColumn("start_date", current_timestamp())
                  .withColumn("end_date", lit(None).cast("timestamp"))
                  .withColumn("is_current", lit(True)))
        scd_df.write.format("delta").mode("overwrite").saveAsTable(scd_table)
        print(resource_name, "- SCD table created (first load):", scd_df.count(), "records")
        print("---------------------------------------------")
        return
    
    existing_df = spark.table(scd_table).filter(col("is_current") == True)
    existing_hashes = existing_df.select("resource_id", col("record_hash").alias("old_hash"))
    comparison_df = new_data_df.join(existing_hashes, on="resource_id", how="left")
    changed_or_new = comparison_df.filter(
        (col("old_hash").isNull()) | (col("record_hash") != col("old_hash"))
    ).drop("old_hash")
    
    if changed_or_new.count() == 0:
        print(resource_name, "- No changes detected.")
        print("---------------------------------------------")
        return
    
    changed_ids = [row["resource_id"] for row in changed_or_new.select("resource_id").collect()]
    full_existing = spark.table(scd_table)
    still_current = full_existing.filter(~col("resource_id").isin(changed_ids))
    now_inactive = (full_existing.filter(col("resource_id").isin(changed_ids))
                     .withColumn("is_current", lit(False))
                     .withColumn("end_date", current_timestamp()))
    new_versions = (changed_or_new.withColumn("start_date", current_timestamp())
                     .withColumn("end_date", lit(None).cast("timestamp"))
                     .withColumn("is_current", lit(True)))
    
    final_df = still_current.unionByName(now_inactive).unionByName(new_versions)
    final_df.write.format("delta").mode("overwrite").saveAsTable(scd_table)
    
    print(resource_name, "- SCD table updated:", changed_or_new.count(), "changed/new records")
    print("---------------------------------------------")

StatementMeta(, 6450f0f5-9c23-4dcf-8a34-8452e5167583, 20, Finished, Available, Finished, False)

In [19]:
def build_gold_layer():
    patient_df = spark.table("silver_patient").withColumn(
        "patient_reference", concat(lit("Patient/"), col("resource_id"))
    )
    encounter_counts = spark.table("silver_encounter").groupBy("patient_reference").agg(spark_count("resource_id").alias("total_encounters"))
    observation_counts = spark.table("silver_observation").groupBy("patient_reference").agg(spark_count("resource_id").alias("total_observations"))
    condition_counts = spark.table("silver_condition").groupBy("patient_reference").agg(spark_count("resource_id").alias("total_conditions"))
    
    gold_df = (patient_df.select("resource_id", "given_name", "family_name", "gender", "birth_date", "patient_reference")
               .join(encounter_counts, on="patient_reference", how="left")
               .join(observation_counts, on="patient_reference", how="left")
               .join(condition_counts, on="patient_reference", how="left")
               .fillna(0, subset=["total_encounters", "total_observations", "total_conditions"]))
    
    gold_df.write.format("delta").mode("overwrite").saveAsTable("gold_patient_summary")
    print("Gold table created:", gold_df.count(), "records")

StatementMeta(, 6450f0f5-9c23-4dcf-8a34-8452e5167583, 21, Finished, Available, Finished, False)

In [20]:
def run_full_pipeline():
    """
    This is the single entry point that the Fabric Data Pipeline will call.
    It automatically fetches only new data since the last run (incremental),
    then runs it through Bronze -> Silver -> SCD -> Gold, in the required order.
    """
    # Automatically determine the incremental window:
    # last 24 hours (in production, this would be "since last pipeline run")
    end_time = datetime.now(timezone.utc).isoformat()
    start_time = (datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)).isoformat()
    run_label = TODAY_DATE
    
    print("===== RUNNING FULL PIPELINE FOR:", run_label, "=====\n")
    
    # Step 1: Bronze (in required order)
    for resource in RESOURCE_ORDER:
        fetch_and_load_bronze(resource, start_time, end_time, run_label)
    
    # Step 2: Silver
    for resource in RESOURCE_ORDER:
        build_silver_layer(resource)
    
    # Step 3: SCD Type 2
    for resource in RESOURCE_ORDER:
        apply_scd_type2(resource)
    
    # Step 4: Gold
    build_gold_layer()
    
    print("\n===== PIPELINE RUN COMPLETE =====")

# Run it once now
run_full_pipeline()

StatementMeta(, 6450f0f5-9c23-4dcf-8a34-8452e5167583, 22, Finished, Available, Finished, False)

===== RUNNING FULL PIPELINE FOR: 2026-09-05 =====

2026-09-05 - Patient - Records found: 73
2026-09-05 - Patient - Bronze table updated: 73 records added
---------------------------------------------
2026-09-05 - Encounter - Records found: 29
2026-09-05 - Encounter - Bronze table updated: 29 records added
---------------------------------------------
2026-09-05 - Observation - Records found: 7
2026-09-05 - Observation - Bronze table updated: 7 records added
---------------------------------------------
2026-09-05 - Condition - Records found: 11
2026-09-05 - Condition - Bronze table updated: 11 records added
---------------------------------------------
Patient - Silver table saved: 73 clean records
---------------------------------------------
Encounter - Silver table saved: 29 clean records
---------------------------------------------
Observation - Silver table saved: 7 clean records
---------------------------------------------
Condition - Silver table saved: 11 clean records
------